In [1]:
import pandas as pd
import numpy as np
import calendar
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.templates.default = "plotly_white"
from sklearn.linear_model import LinearRegression
import seaborn as sns
plot_template = dict(
    layout=go.Layout({
        "font_size": 18,
        "xaxis_title_font_size": 24,
        "yaxis_title_font_size": 24})
)
import sys
sys.path.append('../..') # add parent path to sys.path
import HydroErr as he
from metrics import PPMAE,LPMAE


hydro_stations = [
    'Guide',
    'Xunhua'
]
start_date = '1957-01-01'
end_date = '2019-12-31'


Build CER(MLR) model and estimate natural flow during 1986-2019

In [2]:
for hydro_station in hydro_stations:
    df = pd.read_csv(f'../data/{hydro_station.lower()}_vif_modeling_data_1960-2019.csv',parse_dates=['date'],index_col='date')
    df = df.loc[start_date:end_date]

    target = 'flow'
    ystar_col = 'CER-MLR'
    features = list(df.columns.difference([target]))
    features = list(df.columns.copy())
    features.remove(target)
    print(features)

    mlr_prediction = df.copy()
    
    # 归一化
    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()
    
    # 对特征和目标变量进行归一化
    X_normalized = scaler_X.fit_transform(mlr_prediction[features])
    y_normalized = scaler_y.fit_transform(mlr_prediction[[target]])
    
    # 分割训练数据
    mlr_X = X_normalized[mlr_prediction.index <= '1981-12-31']
    mlr_y = y_normalized[mlr_prediction.index <= '1981-12-31']

    mlr_model = LinearRegression().fit(mlr_X, mlr_y)
    print(f'------{hydro_station}------')
    for val in list(mlr_model.coef_[0]):
        print("{0:.3f}".format(val))
    print("截距: ", mlr_model.intercept_)
    print("系数: ", mlr_model.coef_)

    # 预测并反归一化
    mlr_prediction_normalized = mlr_model.predict(X_normalized)
    mlr_prediction[ystar_col] = scaler_y.inverse_transform(mlr_prediction_normalized)
    mlr_prediction = mlr_prediction.loc[:,[target,ystar_col]]
    mlr_prediction.to_csv(f'../results/vif_mlr_{hydro_station.lower()}.csv')

    mlr_cal = mlr_prediction.loc[start_date:'1981-12-31',:]
    mlr_test = mlr_prediction.loc['1982-01-01':'1985-12-31',:]

    metrics = pd.DataFrame(index=['cal','test'])
    metrics.index.name = 'period'

    metrics['ME'] = [he.me(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.me(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MAE'] = [he.mae(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.mae(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MSE'] = [he.mse(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.mse(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MLE'] = [he.mle(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.mle(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MALE'] = [he.male(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.male(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MSLE'] = [he.msle(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.msle(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MAPE'] = [he.mape(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.mape(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MDE'] = [he.mde(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.mde(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MDAE'] = [he.mdae(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.mdae(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MDSE'] = [he.mdse(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.mdse(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['ED'] = [he.ed(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.ed(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['NED'] = [he.ned(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.ned(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['RMSE'] = [he.rmse(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.rmse(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['RMSLE'] = [he.rmsle(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.rmsle(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['NRMSE_RANGE'] = [he.nrmse_range(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.nrmse_range(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['NRMSE_MEAN'] = [he.nrmse_mean(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.nrmse_mean(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['NRMSE_IQR'] = [he.nrmse_iqr(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.nrmse_iqr(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['IRMSE'] = [he.irmse(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.irmse(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MASE'] = [he.mase(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.mase(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['R_SQUARED'] = [he.r_squared(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.r_squared(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['PEARSON_R'] = [he.pearson_r(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.pearson_r(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['SPEARMAN_R'] = [he.spearman_r(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.spearman_r(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['ACC'] = [he.acc(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.acc(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MAPE'] = [he.mape(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.mape(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MAPD'] = [he.mapd(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.mapd(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MAAPE'] = [he.maape(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.maape(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['SMAPE1'] = [he.smape1(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.smape1(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['SMAPE2'] = [he.smape2(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.smape2(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['D'] = [he.d(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.d(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['D1'] = [he.d1(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.d1(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['DMOD'] = [he.dmod(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.dmod(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['DREL'] = [he.drel(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.drel(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['DR'] = [he.dr(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.dr(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['WATT_M'] = [he.watt_m(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.watt_m(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MB_R'] = [he.mb_r(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.mb_r(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['NSE'] = [he.nse(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.nse(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['NSE_MOD'] = [he.nse_mod(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.nse_mod(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['NSE_REL'] = [he.nse_rel(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.nse_rel(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['KGE_2009'] = [he.kge_2009(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.kge_2009(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['KGE_2012'] = [he.kge_2012(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.kge_2012(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['LM_INDEX'] = [he.lm_index(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.lm_index(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['D1_P'] = [he.d1_p(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.d1_p(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['VE'] = [he.ve(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.ve(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['SA'] = [he.sa(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.sa(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['SC'] = [he.sc(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.sc(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['SID'] = [he.sid(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.sid(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['SGA'] = [he.sga(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.sga(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H1_MHE'] = [he.h1_mhe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h1_mhe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H1_MAHE'] = [he.h1_mahe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h1_mahe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H1_RMSHE'] = [he.h1_rmshe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h1_rmshe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H2_MHE'] = [he.h2_mhe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h2_mhe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H2_MAHE'] = [he.h2_mahe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h2_mahe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H2_RMSHE'] = [he.h2_rmshe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h2_rmshe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H3_MHE'] = [he.h3_mhe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h3_mhe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H3_MAHE'] = [he.h3_mahe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h3_mahe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H3_RMSHE'] = [he.h3_rmshe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h3_rmshe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H4_MHE'] = [he.h4_mhe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h4_mhe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H4_MAHE'] = [he.h4_mahe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h4_mahe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H4_RMSHE'] = [he.h4_rmshe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h4_rmshe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H5_MHE'] = [he.h5_mhe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h5_mhe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H5_MAHE'] = [he.h5_mahe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h5_mahe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H5_RMSHE'] = [he.h5_rmshe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h5_rmshe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H6_MHE'] = [he.h6_mhe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h6_mhe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H6_MAHE'] = [he.h6_mahe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h6_mahe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H6_RMSHE'] = [he.h6_rmshe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h6_rmshe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H7_MHE'] = [he.h7_mhe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h7_mhe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H7_MAHE'] = [he.h7_mahe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h7_mahe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H7_RMSHE'] = [he.h7_rmshe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h7_rmshe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H8_MHE'] = [he.h8_mhe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h8_mhe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H8_MAHE'] = [he.h8_mahe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h8_mahe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H8_RMSHE'] = [he.h8_rmshe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h8_rmshe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H10_MHE'] = [he.h10_mhe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h10_mhe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H10_MAHE'] = [he.h10_mahe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h10_mahe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H10_RMSHE'] = [he.h10_rmshe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h10_rmshe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['G_MEAN_DIFF'] = [he.g_mean_diff(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.g_mean_diff(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MEAN_VAR'] = [he.mean_var(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.mean_var(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['PPMAE'] = [PPMAE(y_true=mlr_cal[target],y_pred=mlr_cal[ystar_col]),PPMAE(y_true=mlr_test[target],y_pred=mlr_test[ystar_col]),]
    metrics['LPMAE'] = [LPMAE(y_true=mlr_cal[target],y_pred=mlr_cal[ystar_col]),LPMAE(y_true=mlr_test[target],y_pred=mlr_test[ystar_col]),]
    print(metrics)
    metrics.to_csv(f'../results/vif_mlr_metrics_{hydro_station.lower()}.csv')


['v10', 'AVG-TEM(C)', 'smlt', 'sro', 'sd', 'ssro', 'tnh_flow']
------Guide------
0.003
0.017
-0.006
-0.039
-0.010
0.017
1.015
截距:  [-0.02182293]
系数:  [[ 0.00314036  0.01655805 -0.00624231 -0.03889765 -0.01020028  0.01663099
   1.01524132]]
                  ME        MAE          MSE       MLE      MALE      MSLE  \
period                                                                       
cal     5.077151e-13  18.755469   733.747408  0.003106  0.035371  0.005359   
test   -2.176433e+00  29.451263  2403.658105 -0.018823  0.043069  0.007030   

            MAPE       MDE       MDAE        MDSE  ...    H8_MHE   H8_MAHE  \
period                                             ...                       
cal     3.420032  3.912922  12.830866  164.664874  ...  0.004797  0.030257   
test    4.059809 -1.570335  15.676552  246.001606  ... -0.014571  0.037611   

        H8_RMSHE   H10_MHE  H10_MAHE  H10_RMSHE  G_MEAN_DIFF  MEAN_VAR  \
period                                                      

Build Extension(MLR) model and estimate natural flow during 1986-2019

In [3]:
for hydro_station in hydro_stations:
    df = pd.read_csv(f'../data/{hydro_station.lower()}_vif_modeling_data_1960-2019.csv',parse_dates=['date'],index_col='date')
    df = df.loc[start_date:end_date]
    # drop tnh_flow
    df = df.drop(columns=['tnh_flow']) # 756
    print(df)

    target = 'flow'
    ystar_col = 'E-MLR'
    features = list(df.columns.difference([target]))
    features = list(df.columns.copy())
    features.remove(target)
    print(features)

    mlr_prediction = df.copy()
    mlr_y = mlr_prediction.loc[start_date:'1981-12-31',target]
    mlr_X = mlr_prediction.loc[start_date:'1981-12-31',features]

    # 归一化
    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()
    X_normalized = scaler_X.fit_transform(mlr_X)
    y_normalized = scaler_y.fit_transform(mlr_y.values.reshape(-1,1)).ravel()

    mlr_model = LinearRegression().fit(X_normalized, y_normalized)
    print(f'------{hydro_station}------')
    for val in list(mlr_model.coef_):
        print("{0:.3f}".format(val))
    print("截距: ", mlr_model.intercept_)
    print("系数: ", mlr_model.coef_)

    # 预测并反归一化
    X_all_normalized = scaler_X.transform(mlr_prediction.loc[start_date:end_date,features])
    mlr_prediction_normalized = mlr_model.predict(X_all_normalized)
    mlr_prediction[ystar_col] = scaler_y.inverse_transform(mlr_prediction_normalized.reshape(-1,1))
    mlr_prediction = mlr_prediction.loc[:,[target,ystar_col]]
    mlr_prediction.to_csv(f'../results/mlr(Extension)_{hydro_station.lower()}.csv')

    mlr_cal = mlr_prediction.loc[start_date:'1981-12-31',:]
    mlr_test = mlr_prediction.loc['1982-01-01':'1985-12-31',:]

    

    metrics = pd.DataFrame(index=['cal','test'])
    metrics.index.name = 'period'
    metrics['ME'] = [he.me(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.me(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MAE'] = [he.mae(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.mae(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MSE'] = [he.mse(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.mse(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MLE'] = [he.mle(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.mle(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MALE'] = [he.male(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.male(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MSLE'] = [he.msle(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.msle(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MAPE'] = [he.mape(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.mape(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MDE'] = [he.mde(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.mde(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MDAE'] = [he.mdae(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.mdae(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MDSE'] = [he.mdse(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.mdse(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['ED'] = [he.ed(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.ed(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['NED'] = [he.ned(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.ned(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['RMSE'] = [he.rmse(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.rmse(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['RMSLE'] = [he.rmsle(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.rmsle(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['NRMSE_RANGE'] = [he.nrmse_range(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.nrmse_range(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['NRMSE_MEAN'] = [he.nrmse_mean(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.nrmse_mean(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['NRMSE_IQR'] = [he.nrmse_iqr(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.nrmse_iqr(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['IRMSE'] = [he.irmse(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.irmse(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MASE'] = [he.mase(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.mase(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['R_SQUARED'] = [he.r_squared(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.r_squared(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['PEARSON_R'] = [he.pearson_r(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.pearson_r(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['SPEARMAN_R'] = [he.spearman_r(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.spearman_r(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['ACC'] = [he.acc(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.acc(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MAPE'] = [he.mape(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.mape(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MAPD'] = [he.mapd(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.mapd(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MAAPE'] = [he.maape(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.maape(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['SMAPE1'] = [he.smape1(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.smape1(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['SMAPE2'] = [he.smape2(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.smape2(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['D'] = [he.d(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.d(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['D1'] = [he.d1(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.d1(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['DMOD'] = [he.dmod(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.dmod(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['DREL'] = [he.drel(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.drel(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['DR'] = [he.dr(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.dr(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['WATT_M'] = [he.watt_m(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.watt_m(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MB_R'] = [he.mb_r(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.mb_r(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['NSE'] = [he.nse(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.nse(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['NSE_MOD'] = [he.nse_mod(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.nse_mod(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['NSE_REL'] = [he.nse_rel(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.nse_rel(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['KGE_2009'] = [he.kge_2009(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.kge_2009(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['KGE_2012'] = [he.kge_2012(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.kge_2012(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['LM_INDEX'] = [he.lm_index(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.lm_index(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['D1_P'] = [he.d1_p(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.d1_p(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['VE'] = [he.ve(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.ve(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['SA'] = [he.sa(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.sa(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['SC'] = [he.sc(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.sc(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['SID'] = [he.sid(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.sid(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['SGA'] = [he.sga(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.sga(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H1_MHE'] = [he.h1_mhe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h1_mhe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H1_MAHE'] = [he.h1_mahe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h1_mahe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H1_RMSHE'] = [he.h1_rmshe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h1_rmshe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H2_MHE'] = [he.h2_mhe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h2_mhe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H2_MAHE'] = [he.h2_mahe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h2_mahe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H2_RMSHE'] = [he.h2_rmshe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h2_rmshe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H3_MHE'] = [he.h3_mhe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h3_mhe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H3_MAHE'] = [he.h3_mahe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h3_mahe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H3_RMSHE'] = [he.h3_rmshe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h3_rmshe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H4_MHE'] = [he.h4_mhe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h4_mhe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H4_MAHE'] = [he.h4_mahe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h4_mahe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H4_RMSHE'] = [he.h4_rmshe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h4_rmshe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H5_MHE'] = [he.h5_mhe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h5_mhe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H5_MAHE'] = [he.h5_mahe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h5_mahe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H5_RMSHE'] = [he.h5_rmshe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h5_rmshe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H6_MHE'] = [he.h6_mhe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h6_mhe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H6_MAHE'] = [he.h6_mahe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h6_mahe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H6_RMSHE'] = [he.h6_rmshe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h6_rmshe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H7_MHE'] = [he.h7_mhe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h7_mhe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H7_MAHE'] = [he.h7_mahe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h7_mahe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H7_RMSHE'] = [he.h7_rmshe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h7_rmshe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H8_MHE'] = [he.h8_mhe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h8_mhe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H8_MAHE'] = [he.h8_mahe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h8_mahe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H8_RMSHE'] = [he.h8_rmshe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h8_rmshe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H10_MHE'] = [he.h10_mhe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h10_mhe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H10_MAHE'] = [he.h10_mahe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h10_mahe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H10_RMSHE'] = [he.h10_rmshe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h10_rmshe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['G_MEAN_DIFF'] = [he.g_mean_diff(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.g_mean_diff(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MEAN_VAR'] = [he.mean_var(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.mean_var(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['PPMAE'] = [PPMAE(y_true=mlr_cal[target],y_pred=mlr_cal[ystar_col]),PPMAE(y_true=mlr_test[target],y_pred=mlr_test[ystar_col]),]
    metrics['LPMAE'] = [LPMAE(y_true=mlr_cal[target],y_pred=mlr_cal[ystar_col]),LPMAE(y_true=mlr_test[target],y_pred=mlr_test[ystar_col]),]
    print(metrics)
    metrics.to_csv(f'../results/mlr(Extension)_metrics_{hydro_station.lower()}.csv')


                 v10  AVG-TEM(C)      smlt       sro        sd      ssro  \
date                                                                       
1960-01-31  0.937511  -12.752986  0.000423  0.000062  0.020468  0.004090   
1960-02-29  0.527554   -8.002374  0.003312  0.000932  0.020473  0.003107   
1960-03-31  0.934352   -2.523280  0.010951  0.002701  0.019050  0.002482   
1960-04-30  0.094989    0.700331  0.020436  0.004004  0.015169  0.002131   
1960-05-31 -0.060234    6.175353  0.028711  0.005169  0.004706  0.001985   
...              ...         ...       ...       ...       ...       ...   
2019-08-31 -0.321948   11.461266  0.003087  0.003291  0.000122  0.014945   
2019-09-30 -0.219753    7.523939  0.021006  0.003460  0.002640  0.019583   
2019-10-31  0.254881    2.215784  0.013888  0.000760  0.006718  0.020596   
2019-11-30  0.746981   -3.226049  0.009707  0.000264  0.015395  0.014721   
2019-12-31  0.760826  -12.046475  0.001102  0.000027  0.016767  0.010785   

           

c:\Users\ZJY\miniconda3\envs\pytorch\Lib\site-packages\HydroErr\HydroErr.py:346: RuntimeWarning: invalid value encountered in log1p
  sim_log = np.log1p(simulated_array)
c:\Users\ZJY\miniconda3\envs\pytorch\Lib\site-packages\HydroErr\HydroErr.py:426: RuntimeWarning: invalid value encountered in log1p
  sim_log = np.log1p(simulated_array)
c:\Users\ZJY\miniconda3\envs\pytorch\Lib\site-packages\HydroErr\HydroErr.py:506: RuntimeWarning: invalid value encountered in log1p
  sim_log = np.log1p(simulated_array)
c:\Users\ZJY\miniconda3\envs\pytorch\Lib\site-packages\HydroErr\HydroErr.py:1041: RuntimeWarning: invalid value encountered in log1p
  return np.sqrt(np.mean(np.power(np.log1p(simulated_array) - np.log1p(observed_array), 2)))
c:\Users\ZJY\miniconda3\envs\pytorch\Lib\site-packages\HydroErr\HydroErr.py:3769: RuntimeWarning: invalid value encountered in log10
  second2 = np.log10(simulated_array) - np.log10(np.mean(simulated_array))
c:\Users\ZJY\miniconda3\envs\pytorch\Lib\site-packages\H

Build Routing(MLR) model and estimate natural flow during 1986-2019

In [4]:
# feature和target进行归一化，对预测结果进行反归一化
for hydro_station in hydro_stations:
    df = pd.read_csv(f'../data/{hydro_station.lower()}_vif_modeling_data_1960-2019.csv',parse_dates=['date'],index_col='date')
    df = df.loc[start_date:end_date]
    
    # preserve tnh_flow and flow
    df = df[['tnh_flow','flow']]

    target = 'flow'
    ystar_col = 'R-MLR'
    features = list(df.columns.difference([target]))
    features = list(df.columns.copy())
    features.remove(target)
    print(features)

    mlr_prediction = df.copy()
    # 归一化
    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()
    X_normalized = scaler_X.fit_transform(mlr_prediction[features])
    y_normalized = scaler_y.fit_transform(mlr_prediction[[target]])

    mlr_y = y_normalized[mlr_prediction.index <= '1981-12-31']
    mlr_X = X_normalized[mlr_prediction.index <= '1981-12-31']

    mlr_model = LinearRegression().fit(mlr_X,mlr_y)
    print(f'------{hydro_station}------')
    for val in list(mlr_model.coef_[0]):
        print("{0:.2f}".format(val))
    print("截距: ", mlr_model.intercept_)
    print("系数: ", mlr_model.coef_)

    # 预测并反归一化
    mlr_prediction_normalized = mlr_model.predict(X_normalized)
    mlr_prediction[ystar_col] = scaler_y.inverse_transform(mlr_prediction_normalized.reshape(-1,1))
    mlr_prediction = mlr_prediction.loc[:,[target,ystar_col]]
    mlr_prediction.to_csv(f'../results/mlr(Routing)_{hydro_station.lower()}.csv')

    mlr_cal = mlr_prediction.loc[start_date:'1981-12-31',:]
    mlr_test = mlr_prediction.loc['1982-01-01':'1985-12-31',:]

    metrics = pd.DataFrame(index=['cal','test'])
    metrics.index.name = 'period'
    metrics['ME'] = [he.me(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.me(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MAE'] = [he.mae(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.mae(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MSE'] = [he.mse(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.mse(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MLE'] = [he.mle(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.mle(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MALE'] = [he.male(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.male(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MSLE'] = [he.msle(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.msle(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MAPE'] = [he.mape(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.mape(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MDE'] = [he.mde(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.mde(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MDAE'] = [he.mdae(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.mdae(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MDSE'] = [he.mdse(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.mdse(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['ED'] = [he.ed(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.ed(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['NED'] = [he.ned(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.ned(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['RMSE'] = [he.rmse(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.rmse(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['RMSLE'] = [he.rmsle(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.rmsle(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['NRMSE_RANGE'] = [he.nrmse_range(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.nrmse_range(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['NRMSE_MEAN'] = [he.nrmse_mean(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.nrmse_mean(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['NRMSE_IQR'] = [he.nrmse_iqr(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.nrmse_iqr(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['IRMSE'] = [he.irmse(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.irmse(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MASE'] = [he.mase(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.mase(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['R_SQUARED'] = [he.r_squared(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.r_squared(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['PEARSON_R'] = [he.pearson_r(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.pearson_r(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['SPEARMAN_R'] = [he.spearman_r(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.spearman_r(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['ACC'] = [he.acc(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.acc(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MAPE'] = [he.mape(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.mape(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MAPD'] = [he.mapd(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.mapd(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MAAPE'] = [he.maape(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.maape(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['SMAPE1'] = [he.smape1(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.smape1(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['SMAPE2'] = [he.smape2(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.smape2(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['D'] = [he.d(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.d(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['D1'] = [he.d1(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.d1(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['DMOD'] = [he.dmod(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.dmod(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['DREL'] = [he.drel(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.drel(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['DR'] = [he.dr(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.dr(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['WATT_M'] = [he.watt_m(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.watt_m(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MB_R'] = [he.mb_r(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.mb_r(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['NSE'] = [he.nse(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.nse(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['NSE_MOD'] = [he.nse_mod(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.nse_mod(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['NSE_REL'] = [he.nse_rel(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]), he.nse_rel(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['KGE_2009'] = [he.kge_2009(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.kge_2009(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['KGE_2012'] = [he.kge_2012(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.kge_2012(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['LM_INDEX'] = [he.lm_index(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.lm_index(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['D1_P'] = [he.d1_p(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.d1_p(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['VE'] = [he.ve(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.ve(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['SA'] = [he.sa(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.sa(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['SC'] = [he.sc(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.sc(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['SID'] = [he.sid(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.sid(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['SGA'] = [he.sga(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.sga(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H1_MHE'] = [he.h1_mhe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h1_mhe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H1_MAHE'] = [he.h1_mahe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h1_mahe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H1_RMSHE'] = [he.h1_rmshe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h1_rmshe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H2_MHE'] = [he.h2_mhe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h2_mhe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H2_MAHE'] = [he.h2_mahe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h2_mahe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H2_RMSHE'] = [he.h2_rmshe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h2_rmshe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H3_MHE'] = [he.h3_mhe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h3_mhe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H3_MAHE'] = [he.h3_mahe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h3_mahe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H3_RMSHE'] = [he.h3_rmshe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h3_rmshe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H4_MHE'] = [he.h4_mhe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h4_mhe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H4_MAHE'] = [he.h4_mahe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h4_mahe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H4_RMSHE'] = [he.h4_rmshe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h4_rmshe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H5_MHE'] = [he.h5_mhe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h5_mhe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H5_MAHE'] = [he.h5_mahe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h5_mahe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H5_RMSHE'] = [he.h5_rmshe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h5_rmshe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H6_MHE'] = [he.h6_mhe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h6_mhe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H6_MAHE'] = [he.h6_mahe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h6_mahe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H6_RMSHE'] = [he.h6_rmshe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h6_rmshe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H7_MHE'] = [he.h7_mhe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h7_mhe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H7_MAHE'] = [he.h7_mahe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h7_mahe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H7_RMSHE'] = [he.h7_rmshe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h7_rmshe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H8_MHE'] = [he.h8_mhe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h8_mhe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H8_MAHE'] = [he.h8_mahe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h8_mahe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H8_RMSHE'] = [he.h8_rmshe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h8_rmshe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H10_MHE'] = [he.h10_mhe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h10_mhe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H10_MAHE'] = [he.h10_mahe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h10_mahe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['H10_RMSHE'] = [he.h10_rmshe(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.h10_rmshe(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['G_MEAN_DIFF'] = [he.g_mean_diff(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.g_mean_diff(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['MEAN_VAR'] = [he.mean_var(observed_array=mlr_cal[target], simulated_array=mlr_cal[ystar_col]),he.mean_var(observed_array=mlr_test[target], simulated_array=mlr_test[ystar_col]),]
    metrics['PPMAE'] = [PPMAE(y_true=mlr_cal[target],y_pred=mlr_cal[ystar_col]),PPMAE(y_true=mlr_test[target],y_pred=mlr_test[ystar_col]),]
    metrics['LPMAE'] = [LPMAE(y_true=mlr_cal[target],y_pred=mlr_cal[ystar_col]),LPMAE(y_true=mlr_test[target],y_pred=mlr_test[ystar_col]),]
    print(metrics)
    metrics.to_csv(f'../results/mlr(Routing)_metrics_{hydro_station.lower()}.csv')


['tnh_flow']
------Guide------
1.05
截距:  [-0.01691802]
系数:  [[1.04852594]]
                  ME        MAE          MSE       MLE      MALE      MSLE  \
period                                                                       
cal     5.813531e-14  21.485343   923.844657  0.007460  0.041740  0.005938   
test    5.610930e+00  31.581127  2588.389927 -0.000972  0.041159  0.005965   

            MAPE       MDE       MDAE        MDSE  ...    H8_MHE   H8_MAHE  \
period                                             ...                       
cal     4.088653  5.496405  15.245610  232.436765  ...  0.008883  0.035878   
test    3.960735  1.996370  15.585496  243.725228  ...  0.001576  0.036406   

        H8_RMSHE   H10_MHE  H10_MAHE  H10_RMSHE  G_MEAN_DIFF  MEAN_VAR  \
period                                                                   
cal     0.057023  0.007460  0.041740   0.077060     1.008729  0.005883   
test    0.061086 -0.000972  0.041159   0.077231     0.998304  0.005964   

  